DAY----------------------------------------------------> 2


In [2]:
import pandas as pd
import numpy as np

In [3]:
# Extract

df = pd.read_csv("../Data/online_retail_cleaned.csv")

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year,Month,Day,Weekday
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009,12,1,Tuesday
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12,1,Tuesday
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12,1,Tuesday
3,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009,12,1,Tuesday
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2009,12,1,Tuesday


In [4]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [5]:
df["TotalSales"] = df["Quantity"] * df["Price"]

In [6]:
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

In [7]:
df = df.sort_values("InvoiceDate")
df.reset_index(drop=True, inplace=True)

In [8]:
df.to_csv("etl_online_retail.csv", index=False)

print("ETL Completed Successfully!")

ETL Completed Successfully!


Feature Engineering

In [9]:
import pandas as pd

df = pd.read_csv("etl_online_retail.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [10]:
df["TotalSales"] = df["Quantity"] * df["Price"]

In [11]:
df["InvoiceMonth"] = df["InvoiceDate"].dt.month

In [12]:
df["InvoiceYear"] = df["InvoiceDate"].dt.year

In [13]:
df["InvoiceDay"] = df["InvoiceDate"].dt.day

In [14]:
df = df.sort_values("InvoiceDate")

df["RollingSales_7"] = (
    df["TotalSales"]
    .rolling(window=7, min_periods=1)
    .mean()
)

In [15]:
df["RollingQty_7"] = (
    df["Quantity"]
    .rolling(7, min_periods=1)
    .mean()
)

In [16]:
df["PurchaseCount"] = (
    df.groupby("Customer ID")["Invoice"]
    .transform("count")
)

In [17]:
df["AverageSpend"] = (
    df.groupby("Customer ID")["TotalSales"]
    .transform("mean")
)

In [18]:
df["LifetimeSales"] = (
    df.groupby("Customer ID")["TotalSales"]
    .transform("sum")
)

In [19]:
df["TotalSales"] = df["Quantity"] * df["Price"]

In [20]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)


In [21]:
rfm = (
    df.groupby("Customer ID")
      .agg({
          "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
          "Invoice": "nunique",
          "TotalSales": "sum"
      })
      .reset_index()
)

rfm.columns = [
    "CustomerID",
    "Recency",
    "Frequency",
    "Monetary"
]

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,529,11,372.86
1,12347.0,2,8,3888.01
2,12348.0,249,4,312.36
3,12349.0,19,3,2635.04
4,12350.0,310,1,294.40


In [22]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5,4,3,2,1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1,2,3,4,5]
)

In [23]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,529,11,372.86,1,5,2,152
1,12347.0,2,8,3888.01,5,4,5,545
2,12348.0,249,4,312.36,2,3,2,232
3,12349.0,19,3,2635.04,5,3,5,535
4,12350.0,310,1,294.40,2,1,2,212


In [24]:
df = df.merge(
    rfm,
    left_on="Customer ID",
    right_on="CustomerID",
    how="left"
)

In [25]:
df.to_csv(
    "online_retail_feature_engineered.csv",
    index=False
)

print("Feature Engineering Completed!")

Feature Engineering Completed!
